# Lab 6: Where is the air turning, and where is it piling up?

**Estimated time:** about 90 minutes

**Difficulty:** mathematics: Developing to Moderate (rates of change on a grid); coding: Introductory to Developing (run and modify); science concepts: Moderate; interpretation: Moderate

## What you will be able to do

1. Read a wind map and say, before any calculation, where air converges, diverges, and rotates.
2. Compute a gradient, a divergence, and a vertical curl from gridded data as differences between neighboring points, and verify one value by hand.
3. Interpret the sign and the size (units of 1/s) of divergence and vorticity in a real storm.
4. Test whether the wind blows along height contours, as the balance between the pressure force and the Coriolis force predicts.

## What this lab assumes

- Slope as a rate of change: change in a quantity divided by the distance over which it changes. No vector calculus is needed; the formulas are given.
- Wind is described by two numbers at each point: u (toward the east) and v (toward the north), in meters per second.

## How this lab works

Each part follows the same pattern: **predict** what you expect, **calculate**
it, **visualize** it, then **explain** what you found. Write your prediction
before you run the code. Being wrong is useful information here; it is the
reason for the prediction step.

**Before you run the first cell:** Colab will show a popup warning that this
notebook "was not authored by Google." Colab shows this for every notebook
opened from GitHub, not just this one, and it does not mean anything is wrong.
Click "Run anyway" to continue.

## Step 0: Set up

In [ ]:
# Install the course library from GitHub. The @v0.3.0 pins this notebook to
# one tagged release, so everyone gets exactly the same code and data.
# Colab starts a fresh environment every session, so this line runs every
# time you open the notebook. That is normal, not an error.

!pip install git+https://github.com/SaharMK22/qi_course.git@v0.3.0 --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import qi_course as qc

# Confirm which version of the library is running.
print("qi_course version:", qc.__version__)

## Step 1: The scientific question

In March 1993 a huge storm swept up the eastern United States. Forecasters
call a storm like this one a deep cyclone: air spiraling in around a center
of low pressure.

> On the day the storm was strongest, where was the air turning, and where was it
> piling up or spreading out? Can we find those places from the winds alone?

You will use daily-average winds and heights on the **850 hPa surface**, roughly
1.5 kilometers above the ground. The data come from a *reanalysis*, which blends
real observations with a weather model to give a complete map on a regular grid.
That regular grid is what allows us to take derivatives.

## Step 2: Predict (before running any more code)

Write short answers in the cell below.

1. A cyclone in the Northern Hemisphere turns **counterclockwise**. Looking down on it,
   will the curl (vorticity) at its center be positive or negative? What would it
   be for a clockwise swirl?
2. Air spirals inward toward a low. Near the center, is the air **converging**
   (piling up) or **diverging** (spreading out)? Will divergence be positive or negative there?
3. Wind speeds here are about 10 meters per second, and grid points are about
   250 kilometers apart. Roughly how big do you expect divergence and vorticity
   to be, in units of 1 per second? (Hint: speed divided by distance.)
4. Sketch where you expect the wind arrows to point relative to the lines of equal height
   (the height contours). Along them, across them, or somewhere in between?

*Double-click this cell and type your predictions here:*

## Step 3: A tiny grid, worked by hand

Before using real data, practice on two small made-up wind fields with only
nine points each. Grid points are 100 km apart. The middle point is the one we
care about. Its east and west neighbors are the points to its right and left;
its north and south neighbors are above and below.

The two rates we need are:

**divergence = (change in u going east) + (change in v going north)**

**curl (vertical) = (change in v going east) - (change in u going north)**

Each "change" is the value on the far side minus the value on the near side, divided by
the distance between them (here 2 x 100 km, since the neighbors are two steps apart).

**Predict first:** in Grid A the air circles the middle counterclockwise; in
Grid B all the air flows inward toward the middle. Which grid has nonzero divergence?
Which has nonzero curl? What sign?

In [ ]:
dx = dy = 100_000      # meters between neighboring grid points
# Rows run north (top) to south (bottom). Columns run west (left) to east (right).

# Grid A: counterclockwise circulation around the middle point.
u_A = np.array([[-5, -5, -5], [ 0,  0,  0], [ 5,  5,  5]], dtype=float)
v_A = np.array([[-5,  0,  5], [-5,  0,  5], [-5,  0,  5]], dtype=float)

# Grid B: air flowing inward toward the middle point.
u_B = np.array([[ 5,  0, -5], [ 5,  0, -5], [ 5,  0, -5]], dtype=float)
v_B = np.array([[-5, -5, -5], [ 0,  0,  0], [ 5,  5,  5]], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
xs, ys = np.meshgrid([-1, 0, 1], [1, 0, -1])
for ax, (u_, v_), name in zip(axes, [(u_A, v_A), (u_B, v_B)], ["Grid A", "Grid B"]):
    ax.quiver(xs, ys, u_, v_, scale=40)
    ax.set_title(name); ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6); ax.set_aspect("equal")
plt.show()

In [ ]:
# Index [row, column]. The middle point is [1, 1].
#   east neighbor [1, 2]   west neighbor [1, 0]
#   north neighbor [0, 1]  south neighbor [2, 1]
def rates_at_middle(u, v):
    du_dx = (u[1, 2] - u[1, 0]) / (2 * dx)   # change in u going east
    dv_dy = (v[0, 1] - v[2, 1]) / (2 * dy)   # change in v going north (north minus south)
    dv_dx = (v[1, 2] - v[1, 0]) / (2 * dx)   # change in v going east
    du_dy = (u[0, 1] - u[2, 1]) / (2 * dy)   # change in u going north
    return du_dx + dv_dy, dv_dx - du_dy      # divergence, curl

for name, (u_, v_) in [("A", (u_A, v_A)), ("B", (u_B, v_B))]:
    d, curl = rates_at_middle(u_, v_)
    print(f"Grid {name}: divergence = {d:+.1e} per second, curl = {curl:+.1e} per second")

Did the signs match your prediction? Notice the size too: winds of only 5 m/s
over 100 km gave rates around 1e-4 per second. That "1e-4" is what real storm
values will look like, which is why we will rescale them in the real data.

## Step 4: The real data

In [ ]:
case = qc.load_case("1993-03-13")
lat, lon, u, v, z = case["lat"], case["lon"], case["u"], case["v"], case["z"]
print(case["date"], "-", case["description"])
print("Grid shape (latitudes x longitudes):", u.shape)
print(f"Latitudes {lat.min():.1f} to {lat.max():.1f} N; longitudes {lon.min()-360:.1f} to {lon.max()-360:.1f}")

# Grid points are a fixed number of DEGREES apart, but a degree of longitude
# covers less ground farther from the equator. Check how much.
dx_m, dy_m = qc.grid_spacing_m(lat, lon)
for target in (20, 40, 60):
    row = int(np.argmin(np.abs(lat - target)))
    print(f"At {lat[row]:.1f} N: east-west spacing {abs(dx_m[row, 5])/1000:.0f} km, north-south spacing {abs(dy_m[row, 5])/1000:.0f} km")

Why does the east-west spacing shrink toward the north while the north-south
spacing does not? (Picture the lines of longitude on a globe.) The library takes
this into account, so you do not have to.

## Step 5: Visualize the winds and heights first

In [ ]:
speed = np.hypot(u, v)    # wind speed from its two components

qc.plot_field_map(lat, lon, speed, u=u, v=v, contours=z, symmetric=False,
                  title="850 hPa wind speed (shading), wind (arrows), height (lines)",
                  label="Wind speed (m/s)")
plt.show()

One caution about the picture: over the high terrain of the western United
States the 850 hPa surface lies below the ground, and the reanalysis fills in
values there. Do not read meaning into features over the Rockies and the
Plateau. Before calculating anything, look at the picture. Where is the center of the
storm? Describe the pattern of arrows around it. Compare with your Step 2 sketch:
how do the arrows sit relative to the height contours?

## Step 6: Calculate divergence and vorticity

In [ ]:
# Both functions work from differences between neighboring grid points,
# exactly like the 3 x 3 practice grid, repeated at every point.
divergence = qc.divergence(u, v, lat, lon)
vorticity = qc.vorticity(u, v, lat, lon)      # the vertical component of the curl

# Real values are tiny numbers like 0.00005 per second, so we show them
# in units of 1e-5 per second (multiply by 100,000).
inner = (slice(2, -2), slice(2, -2))          # ignore the outer rows, where one-sided differences are less reliable
i, j = np.unravel_index(np.argmax(vorticity[inner]), vorticity[inner].shape)
i, j = i + 2, j + 2

print(f"Strongest counterclockwise rotation: {vorticity[i, j]*1e5:.1f} x 1e-5 per second")
print(f"   at latitude {lat[i]:.1f} N, longitude {lon[j]-360:.1f}")
print(f"Divergence at that same point:       {divergence[i, j]*1e5:+.1f} x 1e-5 per second")
print(f"Range of divergence over the map:    {divergence[inner].min()*1e5:+.1f} to {divergence[inner].max()*1e5:+.1f} x 1e-5 per second")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
qc.plot_field_map(lat, lon, vorticity * 1e5, u=u, v=v, ax=axes[0],
                  title="Vorticity (curl): red = counterclockwise", label="x 1e-5 per second")
qc.plot_field_map(lat, lon, divergence * 1e5, u=u, v=v, ax=axes[1],
                  title="Divergence: blue = converging, red = spreading", label="x 1e-5 per second")
plt.tight_layout()
plt.show()

## Step 7: Gradient, and a balance between forces

The height of the 850 hPa surface is high where pressure is high and low where
pressure is low. Its **gradient** (how fast it changes going east or north) tells
us the push on the air. In the large-scale atmosphere that push is nearly balanced
by the Coriolis effect of Earth's rotation, and the wind that results is called the **geostrophic wind**.
It blows **along** the height contours with low heights on its left (Northern Hemisphere):

u_g = -(g / f) x (change in height going north)   v_g = (g / f) x (change in height going east)

where g is gravity and f = 2 x (Earth's rotation rate) x sin(latitude).

In [ ]:
u_g, v_g = qc.geostrophic_wind(z, lat, lon)

# Compare with the real wind, in the part of the map away from the tropics (f gets small near the equator).
rows = lat >= 25
obs_u, obs_v = u[rows].ravel(), v[rows].ravel()
geo_u, geo_v = u_g[rows].ravel(), v_g[rows].ravel()
print(f"Correlation, east-west wind:   {np.corrcoef(geo_u, obs_u)[0, 1]:.2f}")
print(f"Correlation, north-south wind: {np.corrcoef(geo_v, obs_v)[0, 1]:.2f}")
print(f"Typical difference (rms): {np.sqrt(np.mean((geo_u-obs_u)**2 + (geo_v-obs_v)**2)):.1f} m/s")

plt.figure(figsize=(5.2, 5.2))
plt.scatter(geo_u, obs_u, s=8, label="east-west wind (u)")
plt.scatter(geo_v, obs_v, s=8, label="north-south wind (v)")
lims = [-25, 25]
plt.plot(lims, lims, "k--", linewidth=0.8, label="perfect agreement")
plt.xlabel("Geostrophic wind from height (m/s)"); plt.ylabel("Observed wind (m/s)")
plt.legend(); plt.grid(alpha=0.3); plt.axis("equal"); plt.show()

## Step 8: Explain

1. **Rotation.** Where is the strongest vorticity relative to the lowest heights
   and the storm center you identified in Step 5? Was the sign what you predicted
   in Step 2? What would the sign be for a clockwise swirl?
2. **Convergence.** Look at the divergence map. Where is the strongest convergence
   relative to the storm center? Air that piles up cannot pile up forever.
   Where could it go, and what weather does that suggest near the storm?
3. **Size.** A vorticity of 8 x 1e-5 per second is a very small number. Use your
   3 x 3 practice grid to say in plain words what it means for the winds around a point.
4. **Balance.** Does the wind blow along the height contours, as Step 7 predicts?
   Where would you expect the geostrophic wind to be a poor guide, and what is
   the scatter plot telling you about how good the match is overall?
5. **Prediction check.** Compare Step 2 questions 3 and 4 with what you found.
   What did your estimate get right or wrong?

## Optional extension: A tropical cyclone, and why the maps get rough

The library also has daily averages for August 27, 2017, two days after Hurricane
Harvey made landfall on the Texas coast. Make the same two maps and compare them with March 1993.

In [ ]:
harvey = qc.load_case("2017-08-27")
hv = qc.vorticity(harvey["u"], harvey["v"], harvey["lat"], harvey["lon"])
hd = qc.divergence(harvey["u"], harvey["v"], harvey["lat"], harvey["lon"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
qc.plot_field_map(harvey["lat"], harvey["lon"], hv * 1e5, u=harvey["u"], v=harvey["v"], ax=axes[0],
                  title="Vorticity, August 27, 2017", label="x 1e-5 per second")
qc.plot_field_map(harvey["lat"], harvey["lon"], hd * 1e5, u=harvey["u"], v=harvey["v"], ax=axes[1],
                  title="Divergence, August 27, 2017", label="x 1e-5 per second")
plt.tight_layout(); plt.show()

Look for the vorticity maximum near the Texas coast. Is that Harvey's remnant circulation? Then ask why
these maps are harder to read than the 1993 maps. Two ideas to test, not
simply accept: (1) a grid point every 250 km cannot show a structure only tens
of kilometers wide, so the storm may be barely visible; (2) a derivative
compares neighbors, so it emphasizes small, local differences much more than the
smooth large-scale pattern. Smoothing the winds slightly before differencing is
one way to test idea 2.

In [ ]:
def smooth(field):
    # Average each point with its eight neighbors (edges copied outward).
    p = np.pad(field, 1, mode="edge")
    total = sum(p[1 + di: p.shape[0] - 1 + di, 1 + dj: p.shape[1] - 1 + dj]
                for di in (-1, 0, 1) for dj in (-1, 0, 1))
    return total / 9.0

for name, cs in [("March 1993", case), ("August 2017", harvey)]:
    raw = qc.vorticity(cs["u"], cs["v"], cs["lat"], cs["lon"])
    smoothed = qc.vorticity(smooth(cs["u"]), smooth(cs["v"]), cs["lat"], cs["lon"])
    rms_raw = np.sqrt(np.mean(raw[inner] ** 2)) * 1e5
    rms_smooth = np.sqrt(np.mean(smoothed[inner] ** 2)) * 1e5
    print(f"{name}: typical vorticity {rms_raw:.1f} -> {rms_smooth:.1f} (x 1e-5 per second) after smoothing the winds once")

Which case lost more vorticity when the winds were smoothed? What does that say
about how much of each map is large-scale pattern versus small-scale detail?

## If something breaks

The install cell is the one most likely to act up. If a later cell reports
that a function does not exist, use Runtime > Restart session, then run the
cells again from the top. For any other error, do not guess. Copy the exact
error text, note which cell it came from, and send both to your instructor or TA.

---

## Notes for the instructor (draft; delete this cell before sharing with students)

**Status:** first working draft. Time estimates are mine, untested with students.

**Data choice (a decision for you, Dale):** I used NOAA's NCEP/NCAR Reanalysis 1 (daily means, 850 hPa, 2.5 degree grid) rather than GFS model fields. Reanalysis comes as small, clean, regular files and avoids GRIB tools in Colab; GFS is finer but needs extra libraries and gives a forecast rather than a best estimate of the observed state. The cost of reanalysis is coarse resolution and daily averaging.

**Case choice:** the March 1993 storm gives a single, large, clean cyclone. In this case the strongest vorticity is 8.6 x 1e-5 per second near 32.5 N, 82.5 W, and the divergence there is -2.0 x 1e-5 per second (convergence), matching the expected signs. The Harvey case is deliberately included as a harder example (see the extension); at 2.5 degrees a hurricane core is not resolved, so it is a lesson in resolution, not a clean demonstration.

**Checks I ran on the numbers:**
- Gradient function verified against analytic test fields; error is a few parts in ten thousand.
- Geostrophic wind against observed 850 hPa wind (latitudes 25 N and north): correlation 0.93 for u and 0.96 for v.
- Divergence and curl use local east-west and north-south distances. The extra curvature (metric) term is neglected; its largest size here is about 0.4 x 1e-5 per second, about 14 percent of the largest divergence on the map (3.0 x 1e-5 per second). Acceptable for intuition; worth a footnote if students compare to published values.

**Likely misconceptions:** confusing vorticity sign with rotation direction; thinking convergence means strong wind; reading 1e-5 as "zero"; assuming 850 hPa winds equal surface winds; believing geostrophic balance should be exact.

**Honest limitations:** in the extension, one smoothing pass reduces typical vorticity by about 15 percent in March 1993 and about 24 percent in August 2017. The difference is small, so the extension question should be framed as something to test and discuss, not as a clear-cut result. Daily averaging blurs fast-moving features. No coastlines are drawn (to avoid a mapping-library dependency); landmark cities are marked instead. The 850 hPa surface lies over high terrain in the western part of the domain, so values there are not meaningful.

**Open questions for Dale:**
1. Is a clean single-case lab better than letting students choose a date? (That needs a download step and larger files.)
2. Do we want coastlines badly enough to add a mapping library to the Colab install?
3. Should the geostrophic-balance section stay, or is it a separate lab?